<pre>
- Poluição do ar - SO₂ 	(µg/m3)	-> Dióxido de Enxofre 
</pre>


In [1]:
import cdsapi
import os, sys
import xarray as xr
from pyspark.sql import functions as F

In [2]:
# Cria uma conexão SPARK

# Adiciona a pasta raiz do projeto (um ou dois níveis acima) no caminho do Python
sys.path.append(os.path.abspath(os.path.join('..')))  # Ajuste a quantidade de '..' conforme a profundidade da subpasta

# Cria uma conexão Spark 
from spark_utils import get_spark_session # ver em C:\Marco Conti\Projetos\MAIS-v2\spark_utils.py
spark = get_spark_session("MeuNotebook")


c:\Marco Conti\Projetos\MAIS-v2\.venv\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [3]:
dataset = "cams-global-reanalysis-eac4"
request = {
    "variable": [
        "sulphur_dioxide"
    ],
    "pressure_level": ["1000"],
    "date": ["2025-12-01/2025-12-31"],
    "time": ["06:00"],
    "data_format": "netcdf",
    "area": [6, -74, -35, -34]
}

client = cdsapi.Client(
    url = "https://ads.atmosphere.copernicus.eu/api",
    key = "34161618-bf6b-41ca-9272-50b917f789b9"
)

ret_download = client.retrieve(dataset, request).download()

print(f"Download completed: {ret_download}")

2026-07-22 16:03:02,039 INFO Request ID is 8281559d-7109-483d-8843-ea2812ab7b62
2026-07-22 16:03:02,226 INFO status has been updated to accepted
2026-07-22 16:03:52,987 INFO status has been updated to running
2026-07-22 16:04:18,952 INFO status has been updated to successful
                                                                                      

Download completed: 2000b138551fa59d44a9196336b811bd.nc


In [4]:
with xr.open_dataset(f"C:\\Marco Conti\\Projetos\\MAIS-v2\\Poluicao\\{ret_download}"
                    ,engine="netcdf4"
                    ,chunks={"time": 365
                            ,"latitude": 100
                            ,"longitude": 100 }
                    ) as ds:
    # Transforma o Dataset em um Spark Dataframe
    df_dask            = ds.to_dask_dataframe()
    df_dask_c          = df_dask.compute()
    df_dioxido_enxofre = spark.createDataFrame(df_dask_c)

df_dioxido_enxofre.printSchema()
df_dioxido_enxofre.show(10, False)

c:\Marco Conti\Projetos\MAIS-v2\.venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:659: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
c:\Marco Conti\Projetos\MAIS-v2\.venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:936: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
c:\Marco Conti\Projetos\MAIS-v2\.venv\Lib\site-packages\pyspark\sql\pandas\types.py:712: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


root
 |-- valid_time: timestamp (nullable = true)
 |-- pressure_level: double (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- so2: float (nullable = true)

+-------------------+--------------+--------+---------+--------------+
|valid_time         |pressure_level|latitude|longitude|so2           |
+-------------------+--------------+--------+---------+--------------+
|2025-12-01 06:00:00|1000.0        |6.0     |-73.5    |9.624728E-10  |
|2025-12-01 06:00:00|1000.0        |6.0     |-72.75   |1.2539658E-9  |
|2025-12-01 06:00:00|1000.0        |6.0     |-72.0    |3.1582204E-10 |
|2025-12-01 06:00:00|1000.0        |6.0     |-71.25   |1.2732926E-10 |
|2025-12-01 06:00:00|1000.0        |6.0     |-70.5    |1.00499165E-10|
|2025-12-01 06:00:00|1000.0        |6.0     |-69.75   |1.7212187E-10 |
|2025-12-01 06:00:00|1000.0        |6.0     |-69.0    |2.012257E-10  |
|2025-12-01 06:00:00|1000.0        |6.0     |-68.25   |1.5165824E-10 |
|2025-1

In [5]:
# Densidade do ar em condições padrão (20°C, 1013.25 hPa) ~ 1.2041 kg/m3
RHO_AIR_STD = 1.2041  # kg/m3
FATOR_CONVERSAO_SO2 = RHO_AIR_STD * 1e9  # ~ 1.2041e9

drop_cols = ["valid_time", "so2"]

df_dioxido_enxofre_ug_m3 = \
    (df_dioxido_enxofre
        .withColumns({"data_medicao": F.col("valid_time").cast("date")
                     ,"indicador": F.lit("Poluição do ar - SO₂ (µg/m3)") 
                     ,"valor": (F.col("so2") * F.lit(FATOR_CONVERSAO_SO2)).cast("double")
                     ,"unidade_medida": F.lit("µg/m3")})
         .drop(*drop_cols)
    )

In [6]:
# df_dioxido_enxofre_ug_m3.toPandas().to_csv("C:\\Marco Conti\\Projetos\\MAIS-v2\\dados\\EAC4-poluicao\\EAC4_dioxido_enxofre.csv", index=False)

df_dioxido_enxofre_ug_m3.toPandas().to_parquet("C:\\Marco Conti\\Projetos\\MAIS-v2\\dados\\EAC4-poluicao\\EAC4_dioxido_enxofre.parquet")


c:\Marco Conti\Projetos\MAIS-v2\.venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [7]:
df_dioxido_enxofre_parquet = \
    spark.read.parquet("C:\\Marco Conti\\Projetos\\MAIS-v2\\dados\\EAC4-poluicao\\EAC4_dioxido_enxofre.parquet")

df_dioxido_enxofre_parquet.printSchema()
df_dioxido_enxofre_parquet.show(10, False)

root
 |-- pressure_level: double (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- data_medicao: date (nullable = true)
 |-- indicador: string (nullable = true)
 |-- valor: double (nullable = true)
 |-- unidade_medida: string (nullable = true)

+--------------+--------+---------+------------+----------------------------+-------------------+--------------+
|pressure_level|latitude|longitude|data_medicao|indicador                   |valor              |unidade_medida|
+--------------+--------+---------+------------+----------------------------+-------------------+--------------+
|1000.0        |6.0     |-73.5    |2025-12-01  |Poluição do ar - SO₂ (µg/m3)|1.1589134601308615 |µg/m3         |
|1000.0        |6.0     |-72.75   |2025-12-01  |Poluição do ar - SO₂ (µg/m3)|1.5099002439455944 |µg/m3         |
|1000.0        |6.0     |-72.0    |2025-12-01  |Poluição do ar - SO₂ (µg/m3)|0.38028131257306086|µg/m3         |
|1000.0        |6.0    

In [8]:
os.remove(r"C:\Marco Conti\Projetos\MAIS-v2\Poluicao\{file_name}".format(file_name = ret_download))